# 04d · Token-level XLM-R large (E4) · GPU (L4 / A100)

E4 is E2 with `FacebookAI/xlm-roberta-large` instead of XLM-R base, trained at learning rate 1e-5 because the large model can diverge at the base model's rate. Everything else is the same: every token is tagged *original*, *rewritten* or *ai* from train fusion texts with a known human source (`splits/fusion_sources.csv`), fusion training windows contain a rewritten token with probability `focus`, prediction covers the whole text with overlapping windows, and the per-text summaries go to the stacker in `07`.

Why: X4's remaining errors trace to E2's token model, not to how its probabilities are summarised. Missed light rewrites get about 0.45 rewritten probability, and human blogs called fusion carry a confident false spike. A larger model with the same tokenizer targets exactly those token-level misses.

Before running:
- push the repo (`src/hw3`, `configs/tokens_xlmr_large.yaml`, `splits/`), because the first cell clones it;
- pick an **L4 or A100** runtime: about 4–5 h on an L4 and about 2 h on an A100. Both run in bf16; a T4 would fall back to fp16 and take about twice as long as an L4;
- add the Colab secrets `KAGGLE_API_TOKEN` and `HF_TOKEN`.

Progress lives in Google Drive (`DRIVE_ARTIFACTS`, used as `HW3_ARTIFACTS`):
- training saves a checkpoint every `checkpoint_every` micro-batches (`checkpoints/E4.pt`, about 6.7 GB, up to twice that while it is being replaced) and deletes it when training ends;
- after a disconnect, run all cells again: training resumes from the checkpoint, or, if training had already finished, the saved model in `models/E4` only predicts;
- keep `batch_size` and `grad_accum` unchanged between a run and its resume.

`SMOKE = True` is a tiny pipeline check; its outputs go to `smoke/` inside the artifacts folder.

Afterwards copy `preds/E4_*`, `runs/E4*` and `logs/E4*` from Drive (or the zip from the last cell) into the local `artifacts/`, and add `E4` to `members` in `configs/ensemble.yaml`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kerfmanner/ai-text-detection-hw3-ucu-ai.git"
REPO = "ai-text-detection-hw3-ucu-ai"
DRIVE_ARTIFACTS = "/content/drive/MyDrive/hw3/artifacts"
CLOUD = "COLAB_RELEASE_TAG" in os.environ or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if CLOUD and Path.cwd().name != REPO:
    if not Path(REPO).exists():
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir(REPO)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==5.24.0"], check=True)
    sys.path.insert(0, str(Path("src").resolve()))
if "COLAB_RELEASE_TAG" in os.environ:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    os.environ["HW3_ARTIFACTS"] = DRIVE_ARTIFACTS
    os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
CLOUD, os.environ.get("HW3_ARTIFACTS")

In [ ]:
SMOKE = False

import pandas as pd
import torch

from hw3 import config, tokens
from hw3.data import download, splits
from hw3.runs import compare, export_outputs, has_preds, report, val_report
from hw3.sentences import fusion_sources

config.set_smoke(SMOKE)
download()
cfg = config.load_config("tokens_xlmr_large")
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
checkpoint = config.art("checkpoints", f"{cfg['id']}.pt")
gpu, f"checkpoint to resume from: {checkpoint.exists()}", cfg

Trains (or resumes) and predicts. If both prediction files already exist it does nothing; if training finished but prediction did not, it predicts from the saved model. Delete `models/E4` on Drive to force a fresh training run.

In [ ]:
done = all(has_preds(cfg["id"], part) for part in ["val", "test"])
trained = (config.art("models", cfg["id"]) / "config.json").exists() and not checkpoint.exists()
info = {} if done else (tokens.repredict if trained else tokens.run)(cfg)
info.get("train_tokens"), info.get("token_f1_val")

Standalone scores use the same rule as E2 (ai = mean ai probability, fusion = (1 − ai) × smoothed max rewritten probability); the stacker in `07` learns from all summaries.

In [ ]:
rep = report(cfg["id"])
rep["per_class"]

Validation fusion texts with a known source, by share of rewritten words: how often E4 calls them human. E2 called about 47% of the ≤ 5% bucket human.

In [ ]:
sources, text = fusion_sources(), splits()[1].set_index("id").text
r = val_report(cfg["id"])
f = r[(r.label == "fusion") & r.id.isin(sources.index)]
f = f.assign(rewritten=[tokens.word_tags(text[i].split(), "fusion", sources[i]).mean() for i in f.id])
f.groupby(pd.cut(f.rewritten, [-0.001, 0.05, 0.1, 0.2, 1.0]), observed=True).agg(n=("id", "size"), called_human=("pred", lambda p: (p == "human").mean()), mean_rewritten_smax=("rewritten_smax", "mean")).round(3)

In [ ]:
compare(["B0", "E1", "E2", cfg["id"]])

In [ ]:
export_outputs(f"{cfg['id']}_outputs", [f"preds/{cfg['id']}_*", f"runs/{cfg['id']}*", f"logs/{cfg['id']}*"])